# Bitcoin Transaction Fraud Detection: Final Model

This notebook builds the final submission files on its own. Every experiment, validation fold and rejected idea is in `02_feature_engineering_and_modelling_final.ipynb`.

## What it produces

| File | Model | Public ROC-AUC |
|---|---|---|
| `sub2_stack2.csv` | LightGBM + neighbour-score stacking ("stack2"), averaged over 3 seeds | 0.95157 |
| `sub3_blend.csv` | Rank average of plain LightGBM (sub1) and stack2 (sub2) | 0.95094 |

## The method in four steps

1. **Clean the features.** Drop columns that duplicate another column (correlation of about 1).
2. **Build the graph.** Turn the edge list into sparse adjacency matrices (who points to whom, and who is linked to whom ignoring direction).
3. **Stage 1: a fraud score for every transaction.** A gradient-boosting model (`HistGradientBoostingClassifier`) scores every row. Training rows get *out-of-fold* scores (the model that scores a row never saw that row's label), so no label leaks into stage 2.
4. **Stage 2: the final LightGBM.** Its inputs are the original features, the stage-1 score, and five summaries of the stage-1 scores of the transaction's neighbours (mean over predecessors, successors and all neighbours, the maximum, and a 2-hop mean). During training 25% of the neighbour summaries are blanked out, because 25% of test transactions have no known neighbours; the model has to learn to cope with that.

All settings are the ones used for the submitted files (`N_TREES = 150`, so LightGBM uses 300 trees; seeds 0, 1, 2). The final models are trained on **all** labelled rows (time steps 1 to 35) and predict the test rows (steps 36 to 49).

## How to run
Put `train.csv`, `test.csv`, `txs_edgelist.csv` and `sample_submission.csv` in a `data/` folder, start Jupyter from the `notebooks/` folder and choose *Restart & Run All*. The notebook needs `numpy`, `pandas`, `scipy`, `scikit-learn` and `lightgbm` (see `requirements.txt`). It does not need `torch`, `networkx` or `xgboost`.

## 1. Setup

In [1]:
import os, time, warnings
import numpy as np, pandas as pd
import scipy.sparse as sp
from scipy.stats import rankdata, spearmanr
from sklearn.ensemble import HistGradientBoostingClassifier
import lightgbm as lgb

warnings.filterwarnings("ignore")
t_start = time.time()

# Settings used for the submitted files
N_TREES = 150               # LightGBM uses 2 x N_TREES = 300 trees (a tree-count check in notebook 02, section 8.3, showed 200-500 trees score about the same)
FINAL_SEEDS = (0, 1, 2)     # the final scores are the average over these random seeds
P_MASK = 0.25               # share of TRAINING rows whose neighbour features are blanked out (the test set has 25.4% of rows with no known neighbours)
RANDOM_STATE = 42           # fixed seed for the sample used to find duplicate columns

# Look for the data folder next to the notebook. Edit DATA_DIR by hand if yours is elsewhere.
DATA_DIR = None
for _c in ["../data", "./data", "data", "../../data", "."]:
    if os.path.exists(os.path.join(_c, "train.csv")):
        DATA_DIR = _c
        break
if DATA_DIR is None:
    raise FileNotFoundError(f"train.csv not found from {os.getcwd()}. Set DATA_DIR to the folder that contains it.")

OUT_DIR = os.path.dirname(os.path.abspath(DATA_DIR)) if os.path.abspath(DATA_DIR) != os.getcwd() else "."
print("DATA_DIR =", os.path.abspath(DATA_DIR), "| outputs ->", os.path.abspath(OUT_DIR))

DATA_DIR = c:\Users\erina\Downloads\bt4012_kaggle competition\data | outputs -> c:\Users\erina\Downloads\bt4012_kaggle competition


## 2. Data and features

In [2]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
edges = pd.read_csv(f"{DATA_DIR}/txs_edgelist.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

feat_all = [c for c in train.columns if c.startswith("feat_")]
test = test.copy()
if "label" not in test.columns:
    test["label"] = np.nan

# One table with train rows first and test rows after, so that the graph can use both
ALL = pd.concat([train.assign(is_test=0), test.assign(is_test=1, label=np.nan)], ignore_index=True)
N = len(ALL)                                  # total number of transactions
T = ALL["time_step"].values                   # time step of each transaction
Y = ALL["label"].values                       # 1 = illicit, 0 = licit, NaN = unknown (and every test row)
IS_TEST = ALL["is_test"].values == 1
LAB = np.where(~np.isnan(Y) & ~IS_TEST)[0]    # row positions of the labelled training rows: the final models train on these
TEST_POS = np.where(IS_TEST)[0]               # row positions of the test rows: the final models predict these

print(f"rows={N:,}  labelled={len(LAB):,}  test={len(TEST_POS):,}")
print(f"train steps {T[~IS_TEST].min()}-{T[~IS_TEST].max()}, test steps {T[IS_TEST].min()}-{T[IS_TEST].max()}")

rows=157,101  labelled=31,235  test=15,329
train steps 1-35, test steps 36-49


In [3]:
# Remove feature columns that duplicate another column (|correlation| >= 0.99999), found on a fixed random sample of rows
samp = ALL[feat_all].sample(20000, random_state=RANDOM_STATE)
corr = samp.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))   # upper triangle only, so each pair is looked at once
dup_cols = [c for c in upper.columns if (upper[c] >= 0.99999).any()]
feat = [c for c in feat_all if c not in dup_cols]
print(f"dropping {len(dup_cols)} duplicate columns -> {len(feat)} features")

# Feature matrix used by every model (raw values, one row per transaction)
X_raw = ALL[feat].values.astype(np.float32)
print("X_raw:", X_raw.shape)

dropping 8 duplicate columns -> 157 features
X_raw: (157101, 157)


## 3. Graph

In [4]:
# Map transaction IDs to row positions, then keep the edges whose two ends are both in train or test
id2pos = pd.Series(np.arange(N), index=ALL["txId"].values)
es = edges["txId1"].map(id2pos); et = edges["txId2"].map(id2pos)
ok = es.notna() & et.notna()
src = es[ok].astype(int).values; dst = et[ok].astype(int).values
print(f"edges: {len(edges):,} total, {ok.sum():,} between known transactions")

# Sparse N x N adjacency matrices (mostly zeros, so stored sparsely)
A_dir = sp.csr_matrix((np.ones(len(src), dtype=np.float32), (src, dst)), shape=(N, N))
A_dir.sum_duplicates(); A_dir.data[:] = 1; A_dir.setdiag(0); A_dir.eliminate_zeros()   # directed, no duplicates, no self-loops
A_in = A_dir.T.tocsr()                                         # row i = the predecessors of i
A_und = ((A_dir + A_dir.T) > 0).astype(np.float32).tocsr()     # undirected: i and j linked in either direction

und_deg = np.asarray(A_und.sum(1)).ravel()                     # number of neighbours of each transaction

def row_max(M, v, fill=-1.0):
    # For each row of M, the largest value of v among the neighbours; rows with no neighbours get `fill`
    M = M.tocsr(); out = np.full(M.shape[0], fill, dtype=np.float64)
    nz = np.diff(M.indptr) > 0
    if nz.any():
        out[nz] = np.maximum.reduceat(v[M.indices], M.indptr[:-1][nz])
    return out

print(f"transactions with no neighbours at all: {(und_deg == 0).mean():.3f} (train rows {(und_deg[~IS_TEST] == 0).mean():.3f}, test rows {(und_deg[IS_TEST] == 0).mean():.3f})")

edges: 234,355 total, 175,918 between known transactions
transactions with no neighbours at all: 0.025 (train rows 0.000, test rows 0.254)


## 4. Models and stacking functions

In [5]:
def make_hgb(seed):
    # Stage-1 model: scikit-learn gradient boosting with default settings
    return HistGradientBoostingClassifier(random_state=seed)

def make_lgbm(seed):
    # Final model: LightGBM, 2 x N_TREES = 300 trees, learning rate 0.05, 31 leaves, 80% of rows and columns per tree
    return lgb.LGBMClassifier(n_estimators=N_TREES * 2, learning_rate=0.05, num_leaves=31, subsample=0.8,
                              subsample_freq=1, colsample_bytree=0.8, random_state=seed, n_jobs=-1, verbose=-1)

def fit_model(factory, Xtr, ytr, seed):
    # Build a fresh, untrained model and fit it on (Xtr, ytr)
    m = factory(seed)
    m.fit(Xtr, ytr)
    return m

In [6]:
def oof_scores(predict_fn, tr, k=4):
    # predict_fn(fit_rows) -> (N, d) scores for ALL nodes from a model fitted on the labelled rows `fit_rows`.
    # Returns stage-1 scores that are safe to use as features:
    #   - training rows get OUT-OF-FOLD scores (from models that never saw their label)
    #   - all other rows (test, unlabelled) get the average of the k models' scores
    steps = np.unique(T[tr]); blocks = np.array_split(steps, min(k, len(steps)))   # k blocks of consecutive time steps
    in_tr = np.zeros(N, bool); in_tr[tr] = True
    out = acc = None
    for b in blocks:
        hold = tr[np.isin(T[tr], b)]; fit = tr[~np.isin(T[tr], b)]     # score block b with a model fitted on the other blocks
        P = np.asarray(predict_fn(fit), dtype=np.float64).reshape(N, -1)
        if out is None:
            out = np.zeros_like(P); acc = np.zeros_like(P)
        out[hold] = P[hold]
        acc[~in_tr] += P[~in_tr]
    out[~in_tr] = acc[~in_tr] / len(blocks)
    return out

def mean_over(M, vec):
    # For each node, the mean of `vec` over its neighbours in the graph M. Nodes with no neighbours get -1.
    d = np.asarray(M.sum(1)).ravel()
    return np.where(d > 0, (M @ vec) / np.maximum(d, 1), -1.0)

def nbr_aggregate(p):
    # Five neighbourhood summaries of the score p (one value per node):
    # mean over predecessors, mean over successors, mean over all neighbours, max over neighbours, 2-hop mean
    p = np.asarray(p, dtype=np.float64)
    m_in, m_out, m_und = mean_over(A_in, p), mean_over(A_dir, p), mean_over(A_und, p)
    mx = row_max(A_und, p)
    m2 = mean_over(A_und, np.where(m_und >= 0, m_und, 0.0))     # mean of the neighbours' means
    return np.column_stack([m_in, m_out, m_und, mx, m2])

def tree_stage1(factory, X):
    # Stage-1 scorer: f(fit_rows, seed) -> (N, 1) array with the fraud probability of every node
    def f(fit, seed):
        y = Y[fit].astype(int)
        m = fit_model(factory, X[fit], y, seed)
        return m.predict_proba(X)[:, 1:2]
    return f

## 5. The two final models

Both are written as `fp(tr, te, seed) -> scores for the rows te`, fitted on the rows `tr`.

* `plain_fp`: LightGBM on the raw features (sub1, used only for the blend).
* `stack2_fp`: the two-stage model (sub2). Steps inside:
  1. out-of-fold stage-1 scores `S` for every node,
  2. neighbour summaries `agg` of `S`,
  3. transactions that really have no neighbours get missing values (`NaN`) instead of `-1`,
  4. a random 25% of the *training* rows also get their neighbour summaries blanked
  5. final LightGBM on `[raw features, own stage-1 score, 5 neighbour summaries]`.

In [7]:
def plain_fp(X, tr, te, seed):
    # Plain LightGBM: fit on rows tr, return the fraud probability of rows te
    m = fit_model(make_lgbm, X[tr], Y[tr].astype(int), seed)
    return m.predict_proba(X[te])[:, 1]

def stack2_fp(tr, te, seed, k=4, p_mask=P_MASK):
    # 1) stage 1: out-of-fold fraud score for every node (4 HistGradientBoosting fits)
    stage1 = tree_stage1(make_hgb, X_raw)
    S = oof_scores(lambda fit: stage1(fit, seed), tr, k)
    # 2) neighbour summaries of the stage-1 score
    agg = nbr_aggregate(S[:, 0])
    agg[und_deg == 0] = np.nan                       # real isolates: missing, not "-1"
    # 3) blank the neighbour summaries of a random share of the TRAINING rows
    rng = np.random.RandomState(seed)
    mask = np.zeros(N, bool)
    mask[tr] = rng.rand(len(tr)) < p_mask
    agg[mask] = np.nan
    # 4) final LightGBM on [raw features, own stage-1 score, neighbour summaries]
    X = np.hstack([X_raw, S, agg]).astype(np.float32)
    return plain_fp(X, tr, te, seed)

## 6. Fit on all labelled rows and predict the test rows

Each model is fitted once per seed (0, 1, 2) on all labelled rows `LAB` and scores the test rows `TEST_POS`. The three score vectors are averaged.

In [8]:
def predict_test(name, fp):
    # Average the test predictions of fp over FINAL_SEEDS
    t0 = time.time(); preds = []
    for sd in FINAL_SEEDS:
        preds.append(np.asarray(fp(LAB, TEST_POS, sd), dtype=np.float64))
        print(f"  {name}: seed {sd} done ({time.time() - t0:.0f}s)")
    return np.mean(preds, axis=0)

s1 = predict_test("plain LightGBM", lambda tr, te, sd: plain_fp(X_raw, tr, te, sd))
s2 = predict_test("stack2", stack2_fp)
print("spearman correlation of the two score vectors:", round(spearmanr(s1, s2)[0], 4))

  plain LightGBM: seed 0 done (4s)
  plain LightGBM: seed 1 done (12s)
  plain LightGBM: seed 2 done (20s)
  stack2: seed 0 done (38s)
  stack2: seed 1 done (69s)
  stack2: seed 2 done (106s)
spearman correlation of the two score vectors: 0.9475


## 7. Write the submission files

ROC-AUC depends only on the ordering of the scores, so every score vector is converted to ranks in (0, 1] first. This also puts the two models on the same scale before they are averaged for `sub3`. Each file keeps the exact row order of `sample_submission.csv`.

In [9]:
def rank01(s):
    # Scores -> ranks scaled to (0, 1]; the highest score gets 1.0
    return rankdata(s) / len(s)

def write_submission(scores, path):
    sub = sample_sub.copy()
    sub["target"] = scores
    assert len(sub) == len(sample_sub) and (sub["index"] == sample_sub["index"]).all(), "row order differs from sample_submission.csv"
    assert len(scores) == len(TEST_POS), "one score per test row expected"
    assert not sub["target"].isna().any(), "NaN in the scores"
    assert sub["target"].between(0, 1).all(), "scores must lie in [0, 1]"
    sub.to_csv(path, index=False)
    print(f"wrote {path}: rows={len(sub)} range=({sub['target'].min():.4f}, {sub['target'].max():.4f})")
    return sub

# sample_submission.csv has columns index, target. Its 'index' must equal the 'index' column of test.csv, row by row.
assert (sample_sub["index"].values == ALL.loc[TEST_POS, "index"].values).all(), "sample_submission order differs from test.csv"

# Keep any submission files already on disk
OLD = {f: pd.read_csv(f"{OUT_DIR}/{f}") for f in ["sub2_stack2.csv", "sub3_blend.csv"] if os.path.exists(f"{OUT_DIR}/{f}")}

sub2 = write_submission(rank01(s2), f"{OUT_DIR}/sub2_stack2.csv")                    # final entry 1: stack2
sub3 = write_submission(np.mean([rank01(s1), rank01(s2)], axis=0), f"{OUT_DIR}/sub3_blend.csv")   # final entry 2: rank average of plain + stack2
print(f"total run time: {(time.time() - t_start) / 60:.1f} minutes")

wrote c:\Users\erina\Downloads\bt4012_kaggle competition/sub2_stack2.csv: rows=15329 range=(0.0001, 1.0000)
wrote c:\Users\erina\Downloads\bt4012_kaggle competition/sub3_blend.csv: rows=15329 range=(0.0001, 0.9999)
total run time: 2.3 minutes


## 8. Sanity checks

In [10]:
for f in ["sub2_stack2.csv", "sub3_blend.csv"]:
    d = pd.read_csv(f"{OUT_DIR}/{f}")
    print(f"{f}: {d.shape}, columns {list(d.columns)}, unique scores {d['target'].nunique():,}")
print(pd.read_csv(f"{OUT_DIR}/sub2_stack2.csv").head())

# Share of test rows with no neighbours: the stage-2 model sees NaN neighbour summaries for these rows
print("test rows with no neighbours: %.3f" % (und_deg[TEST_POS] == 0).mean())

# Optional: compare with files that were on disk before this run (e.g. written by notebook 02)
for f, old in OLD.items():
    new = pd.read_csv(f"{OUT_DIR}/{f}")
    print(f"{f}: Spearman with the previous file = {spearmanr(old['target'], new['target'])[0]:.6f}, "
          f"max abs rank difference = {np.abs(old['target'] - new['target']).max():.6f}")

sub2_stack2.csv: (15329, 2), columns ['index', 'target'], unique scores 15,326
sub3_blend.csv: (15329, 2), columns ['index', 'target'], unique scores 12,333
   index    target
0      0  0.553135
1      1  0.672581
2      2  0.754061
3      3  0.584970
4      4  0.336747
test rows with no neighbours: 0.254
sub2_stack2.csv: Spearman with the previous file = 1.000000, max abs rank difference = 0.000000
sub3_blend.csv: Spearman with the previous file = 1.000000, max abs rank difference = 0.000000
